# Lab 1 · Your first DuckDB query

**Due:** Sunday, October 25 at 11:59 PM

In this lab you will:
1. Connect to a sample dataset with DuckDB
2. Run your first SQL queries
3. Record three observations about the data

This is the **in-class** part of Lab 1. The at-home part (your own data, plus a lifecycle map) is described in `labs/lab1.md`.

**How to run a cell:** click inside it and press **Shift + Enter**. Run the cells in order, top to bottom.
The first time, VS Code may ask you to **Select Kernel** → choose **Python Environments** → the Python 3.12 option.

## 1. Connect to DuckDB

DuckDB is a database that runs *inside* your Python session: there's no server to start and no password.
The cell below opens a database file called `lab1.duckdb` in your repo (it's created the first time you run this).

In [13]:
from pathlib import Path
import duckdb

# Find the repo folder, whether this notebook runs from the repo root or from notebooks/
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA = ROOT / "data" / "penguins.csv"

con = duckdb.connect(str(ROOT / "lab1.duckdb"))
print("Connected. Sample data at:", DATA.relative_to(ROOT))

Connected. Sample data at: data/penguins.csv


## 2. Look at the raw file

DuckDB can query a CSV file directly, before it's loaded into a table. `LIMIT 5` shows only the first five rows.

In [14]:
con.sql(f"SELECT * FROM '{DATA}' LIMIT 5")

┌─────────┬───────────┬────────────────┬───────────────┬───────────────────┬─────────────┬─────────┬───────┐
│ species │  island   │ bill_length_mm │ bill_depth_mm │ flipper_length_mm │ body_mass_g │   sex   │ year  │
│ varchar │  varchar  │    varchar     │    varchar    │      varchar      │   varchar   │ varchar │ int64 │
├─────────┼───────────┼────────────────┼───────────────┼───────────────────┼─────────────┼─────────┼───────┤
│ Adelie  │ Torgersen │ 39.1           │ 18.7          │ 181               │ 3750        │ male    │  2007 │
│ Adelie  │ Torgersen │ 39.5           │ 17.4          │ 186               │ 3800        │ female  │  2007 │
│ Adelie  │ Torgersen │ 40.3           │ 18            │ 195               │ 3250        │ female  │  2007 │
│ Adelie  │ Torgersen │ NA             │ NA            │ NA                │ NA          │ NA      │  2007 │
│ Adelie  │ Torgersen │ 36.7           │ 19.3          │ 193               │ 3450        │ female  │  2007 │
└─────────┴────────

## 3. Load it into a table

Now copy the file into a proper table called `penguins`. Once it's a table, it lives in `lab1.duckdb` and you can query it by name.
`CREATE OR REPLACE` means you can rerun this cell safely: it replaces the table rather than failing or duplicating it.

In [15]:
con.sql(f"CREATE OR REPLACE TABLE penguins AS SELECT * FROM read_csv('{DATA}')")
con.sql("SELECT count(*) AS row_count FROM penguins")

┌───────────┐
│ row_count │
│   int64   │
├───────────┤
│       344 │
└───────────┘

## 4. What did DuckDB decide about each column?

`DESCRIBE` shows each column's name and the **type** DuckDB guessed: `VARCHAR` is text, `BIGINT` is a whole number, `DOUBLE` is a decimal.
Look closely: does every guess make sense?

In [16]:
con.sql("DESCRIBE penguins")

┌───────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name    │ column_type │  null   │   key   │ default │  extra  │
│      varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ species           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ island            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ bill_length_mm    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ bill_depth_mm     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ flipper_length_mm │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ body_mass_g       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ sex               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ year              │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
└───────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

### Something's wrong

`bill_length_mm` and `body_mass_g` are measurements, but DuckDB stored them as **text** (`VARCHAR`). Why?

Search the raw file for the letters `NA`: whoever recorded the data wrote `NA` where a value was missing. DuckDB saw letters in a number column and played it safe by treating the whole column as text. Text can't be averaged, so every calculation on these columns would fail.

This is the course in miniature: **the data arrived, but it wasn't ready.** The fix is to tell DuckDB that `NA` means "missing" (a `NULL` in database terms), then reload.

In [17]:
con.sql(f"""
    CREATE OR REPLACE TABLE penguins AS
    SELECT * FROM read_csv('{DATA}', nullstr = 'NA')
""")
con.sql("DESCRIBE penguins")

┌───────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name    │ column_type │  null   │   key   │ default │  extra  │
│      varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ species           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ island            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ bill_length_mm    │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ bill_depth_mm     │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ flipper_length_mm │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ body_mass_g       │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ sex               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ year              │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
└───────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

Now the measurements are numbers (`DOUBLE` and `BIGINT`) and the missing values are real `NULL`s.

## 5. Ask questions with SQL

How many penguins of each species were measured, and how heavy are they on average?

In [18]:
con.sql("""
    SELECT species,
           count(*)                   AS penguins,
           round(avg(body_mass_g), 0) AS avg_mass_g
    FROM penguins
    GROUP BY species
    ORDER BY penguins DESC
""")

┌───────────┬──────────┬────────────┐
│  species  │ penguins │ avg_mass_g │
│  varchar  │  int64   │   double   │
├───────────┼──────────┼────────────┤
│ Adelie    │      152 │     3701.0 │
│ Gentoo    │      124 │     5076.0 │
│ Chinstrap │       68 │     3733.0 │
└───────────┴──────────┴────────────┘

Which values are missing? `count(column)` skips empty values, so subtracting it from `count(*)` gives the number missing.

In [19]:
con.sql("""
    SELECT count(*) - count(sex)          AS missing_sex,
           count(*) - count(body_mass_g)  AS missing_body_mass,
           count(*) - count(bill_length_mm) AS missing_bill_length
    FROM penguins
""")

┌─────────────┬───────────────────┬─────────────────────┐
│ missing_sex │ missing_body_mass │ missing_bill_length │
│    int64    │       int64       │        int64        │
├─────────────┼───────────────────┼─────────────────────┤
│          11 │                 2 │                   2 │
└─────────────┴───────────────────┴─────────────────────┘

## 6. Your turn

Write **one query of your own** in the cell below. Some ideas:
- Which island has the most penguins? (`GROUP BY island`)
- Which species has the longest average flipper?
- How many penguins were measured each `year`?

Stuck? Copy a query from above and change one thing.

In [20]:
con.sql("""
    -- write your query here
    SELECT *
    FROM penguins
    LIMIT 5
""")

┌─────────┬───────────┬────────────────┬───────────────┬───────────────────┬─────────────┬─────────┬───────┐
│ species │  island   │ bill_length_mm │ bill_depth_mm │ flipper_length_mm │ body_mass_g │   sex   │ year  │
│ varchar │  varchar  │     double     │    double     │       int64       │    int64    │ varchar │ int64 │
├─────────┼───────────┼────────────────┼───────────────┼───────────────────┼─────────────┼─────────┼───────┤
│ Adelie  │ Torgersen │           39.1 │          18.7 │               181 │        3750 │ male    │  2007 │
│ Adelie  │ Torgersen │           39.5 │          17.4 │               186 │        3800 │ female  │  2007 │
│ Adelie  │ Torgersen │           40.3 │          18.0 │               195 │        3250 │ female  │  2007 │
│ Adelie  │ Torgersen │           NULL │          NULL │              NULL │        NULL │ NULL    │  2007 │
│ Adelie  │ Torgersen │           36.7 │          19.3 │               193 │        3450 │ female  │  2007 │
└─────────┴────────

Q1: Which island has the most penguins?
A1: Biscoe

In [21]:
con.sql("""
SELECT island, COUNT(*) AS penguin_count
FROM penguins
GROUP BY island
ORDER BY penguin_count DESC
""")


┌───────────┬───────────────┐
│  island   │ penguin_count │
│  varchar  │     int64     │
├───────────┼───────────────┤
│ Biscoe    │           168 │
│ Dream     │           124 │
│ Torgersen │            52 │
└───────────┴───────────────┘

Q2: Which species has the longest average flipper?
A2: Gentoo

In [22]:
con.sql("""
SELECT species, AVG(flipper_length_mm) AS avg_flipper_length
FROM penguins
GROUP BY species
ORDER BY avg_flipper_length DESC
""")


┌───────────┬────────────────────┐
│  species  │ avg_flipper_length │
│  varchar  │       double       │
├───────────┼────────────────────┤
│ Gentoo    │  217.1869918699187 │
│ Chinstrap │  195.8235294117647 │
│ Adelie    │ 189.95364238410596 │
└───────────┴────────────────────┘

Q3: How many penguins were measured each `year`?
A3: 2007:110; 2008:114; 2009:120

## 7. Three observations

Double-click this cell to edit it. Write three things you noticed about the data. Good observations are specific and point to something you'd want to check before trusting the data.

1.Biscoe Island has the most penguins

2.Gentoo has the longest average flipper

3.Penguins measures increase every year

*Example:* "11 penguins have no recorded sex. Before comparing males and females, I'd need to decide whether to drop them or keep them as 'unknown'." 

## 8. Save your work

1. Close the connection by running the cell below.
2. Commit and push: open **Source Control** in the left sidebar, type a message like `Lab 1 in class`, click **Commit**, then **Sync Changes**.

Your `lab1.duckdb` file is not uploaded. That's intentional: the database can always be rebuilt from this notebook.

**Next:** the at-home part is in `notebooks/lab1_home_your_data.ipynb`, where you do all of this again with a dataset you choose. See `labs/lab1.md` for the full Lab 1 checklist.

In [24]:
con.close()
print("Connection closed. Don't forget to commit and push.")

Connection closed. Don't forget to commit and push.


Bad pipe message: %s [b' q=0.9, image/jxl, image/avif, image/webp, image/apng, */*; q=0.8, application/signed-exchange; v=b3; q=0.7', b'Host: localh']
Bad pipe message: %s [b't:46859\r\nUser-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/1', b'.0.0.0 Safari/537.36\r\nAccept-Encoding: gzip, deflate', b'br, zstd\r\nAccept-Language: zh-CN, zh; q=0.9']
Bad pipe message: %s [b'Cache-Contro']
Bad pipe message: %s [b' max-age=0\r\nReferer: https://github.com/\r\nX-Request-ID: 7d10ad0534c052ad29dd2efb59f6b91f\r\nX-Real-IP: 10.241']
Bad pipe message: %s [b'.36\r\nX-Forwarded-Port: 443\r\nX-Forwarded-Schem']
Bad pipe message: %s [b' https\r\nX-Original-URI: /\r\nX-Scheme: https\r\nsec-fetch-site: cross-site\r\nsec-fetch-mode: navigate\r\nse']
Bad pipe message: %s [b'fetch-dest: document\r\nsec-ch-ua: "Google Chrome";v="155", "Chromium";v="155", "Not(A:Brand";v="24"', b'sec-ch-ua-mo']
Bad pipe message: %s [b'le: ?0\r\nsec-ch-ua-platform: "Windows"\r